# Stability-Guided Ensemble Pruning for Improved Generalization

This notebook implements a stability-guided ensemble pruning pipeline that:
1. Evaluates individual classifier stability via repeated cross-validation
2. Prunes unstable/weak models based on ROC-AUC mean and variance
3. Compares pruned ensembles against full ensembles and individual models
4. Tunes only the pruned subset for final performance

**Datasets:** Heart Disease (UCI) and Pima Indians Diabetes

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import (
    make_scorer, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    matthews_corrcoef, confusion_matrix
)
from xgboost import XGBClassifier

from ucimlrepo import fetch_ucirepo

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- Output directories ---
DATASETS_DIR = 'datasets'
PLOTS_DIR = 'plots'
os.makedirs(DATASETS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# --- Custom scorers ---
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0

SCORING = {
    'accuracy': 'accuracy',
    'recall': 'recall',
    'precision': 'precision',
    'specificity': make_scorer(specificity_score),
    'f1_weighted': 'f1_weighted',
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision',
    'mcc': make_scorer(matthews_corrcoef)
}

print('All imports successful.')
print(f'Output directories: {DATASETS_DIR}/, {PLOTS_DIR}/')

All imports successful.
Output directories: datasets/, plots/


## 1. Load Datasets

In [2]:
# --- Heart Disease (UCI) ---
heart_repo = fetch_ucirepo(id=45)
heart_X = heart_repo.data.features.copy()
heart_y = heart_repo.data.targets.copy()

# The target have values 0-4; binarise: 0 = no disease, 1 = disease
heart_y.columns = ['target']
heart_y['target'] = (heart_y['target'] > 0).astype(int)

heart_df = pd.concat([heart_X, heart_y], axis=1)
heart_df.to_csv(f'{DATASETS_DIR}/heart_disease_clean.csv', index=False)
print(f'Heart Disease shape: {heart_df.shape}')
print(f'Class distribution:\n{heart_df["target"].value_counts()}')
print(f'Features: {list(heart_X.columns)}')
print()

Heart Disease shape: (303, 14)
Class distribution:
target
0    164
1    139
Name: count, dtype: int64
Features: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']



In [3]:
# --- Pima Indians Diabetes ---
pima_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
             'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
pima_df = pd.read_csv(
    'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv',
    header=None, names=pima_cols
)
pima_df.to_csv(f'{DATASETS_DIR}/pima_diabetes_clean.csv', index=False)
print(f'Pima Diabetes shape: {pima_df.shape}')
print(f'Class distribution:\n{pima_df["Outcome"].value_counts()}')
print(f'Features: {pima_cols[:-1]}')
print()

Pima Diabetes shape: (768, 9)
Class distribution:
Outcome
0    500
1    268
Name: count, dtype: int64
Features: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']



## 2. Preprocessing

- **Heart Disease**: `SimpleImputer(median)` + `StandardScaler` via sklearn `Pipeline`
- **Pima Diabetes**: `IterativeImputer` + `StandardScaler` + `SMOTE` via imblearn `Pipeline`

All preprocessing is performed **inside each CV fold** to prevent data leakage.
SMOTE is applied only to training folds via the imblearn pipeline.

In [4]:
# Prepare X, y for each dataset
# Heart Disease
heart_X = heart_df.drop('target', axis=1)
heart_y = heart_df['target'].values

# Replace any zero values in columns where 0 is physiologically impossible with NaN
# (Heart dataset may have missing values encoded as NaN already from UCI)
print('Heart Disease missing values per column:')
print(heart_X.isnull().sum())
print()

# Pima Diabetes — zeros in Glucose, BloodPressure, SkinThickness, Insulin, BMI are missing
pima_X = pima_df.drop('Outcome', axis=1).copy()
pima_y = pima_df['Outcome'].values

zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
pima_X[zero_cols] = pima_X[zero_cols].replace(0, np.nan)
print('Pima Diabetes missing values per column (after replacing 0s):')
print(pima_X.isnull().sum())
print()

# Check class balance
for name, y in [('Heart', heart_y), ('Pima', pima_y)]:
    counts = np.bincount(y)
    ratio = counts.min() / counts.max()
    print(f'{name}: class 0 = {counts[0]}, class 1 = {counts[1]}, minority ratio = {ratio:.2f}')
    if ratio < 0.5:
        print(f'  -> Imbalanced! Will include PR-AUC in evaluation.')
    else:
        print(f'  -> Reasonably balanced.')
print()

Heart Disease missing values per column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
dtype: int64

Pima Diabetes missing values per column (after replacing 0s):
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
dtype: int64

Heart: class 0 = 164, class 1 = 139, minority ratio = 0.85
  -> Reasonably balanced.
Pima: class 0 = 500, class 1 = 268, minority ratio = 0.54
  -> Reasonably balanced.



## 3. Define Classifier Pool (Default Hyperparameters)

All classifiers use **default** hyperparameters. This is intentional to ensure a fair, unbiased,
leakage-free stability comparison across all models.

In [5]:
def get_base_classifiers():
    """Return dict of name -> classifier with default hyperparameters."""
    return {
        'LR': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
        'SVM': SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
        'kNN': KNeighborsClassifier(n_neighbors=5),
        'RF': RandomForestClassifier(random_state=RANDOM_STATE),
        'XGBoost': XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss',
                                  use_label_encoder=False)
    }

def make_heart_pipeline(clf):
    """Heart Disease pipeline: SimpleImputer + StandardScaler."""
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', clf)
    ])

def make_pima_pipeline(clf):
    """Pima Diabetes pipeline: IterativeImputer + StandardScaler + SMOTE."""
    return ImbPipeline([
        ('imputer', IterativeImputer(max_iter=10, random_state=RANDOM_STATE)),
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('clf', clf)
    ])

def make_pipeline(clf, dataset_name):
    """Dataset-aware pipeline builder."""
    if dataset_name == 'pima':
        return make_pima_pipeline(clf)
    return make_heart_pipeline(clf)

def make_heart_ensemble_pipeline(ensemble):
    """Heart Disease ensemble pipeline."""
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('ensemble', ensemble)
    ])

def make_pima_ensemble_pipeline(ensemble):
    """Pima Diabetes ensemble pipeline: IterativeImputer + Scaler + SMOTE + ensemble."""
    return ImbPipeline([
        ('imputer', IterativeImputer(max_iter=10, random_state=RANDOM_STATE)),
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('ensemble', ensemble)
    ])

def make_ensemble_pipeline(ensemble, dataset_name):
    """Dataset-aware ensemble pipeline builder."""
    if dataset_name == 'pima':
        return make_pima_ensemble_pipeline(ensemble)
    return make_heart_ensemble_pipeline(ensemble)

print('Classifier pool defined: LR, SVM, kNN, RF, XGBoost')
print('Heart pipeline: SimpleImputer(median) -> StandardScaler -> clf')
print('Pima pipeline:  IterativeImputer -> StandardScaler -> SMOTE -> clf')

Classifier pool defined: LR, SVM, kNN, RF, XGBoost
Heart pipeline: SimpleImputer(median) -> StandardScaler -> clf
Pima pipeline:  IterativeImputer -> StandardScaler -> SMOTE -> clf


## 4. Stage 1 — Stability Evaluation (Default Params)

RepeatedStratifiedKFold (5 splits x 5 repeats = 25 scores per model).  
Scaler is fit **only on training data** per fold via Pipeline.

In [6]:
def extract_metrics(cv_results):
    """Extract all metric arrays from cross_validate results."""
    return {
        'accuracy': cv_results['test_accuracy'],
        'recall': cv_results['test_recall'],
        'precision': cv_results['test_precision'],
        'specificity': cv_results['test_specificity'],
        'f1_weighted': cv_results['test_f1_weighted'],
        'roc_auc': cv_results['test_roc_auc'],
        'pr_auc': cv_results['test_pr_auc'],
        'mcc': cv_results['test_mcc'],
    }

def metrics_to_row(name_key, name_val, metrics):
    """Convert metric arrays to a summary row dict."""
    auc = metrics['roc_auc']
    stability = auc.mean() / (auc.std() + 1e-9)
    row = {name_key: name_val}
    for metric_name, arr in metrics.items():
        nice = (metric_name.replace('_', ' ').title()
                .replace('Roc Auc', 'AUC').replace('Pr Auc', 'PR-AUC')
                .replace('F1 Weighted', 'F1').replace('Mcc', 'MCC'))
        row[f'Mean {nice}'] = arr.mean()
        row[f'Std {nice}'] = arr.std()
    row['Stability Score'] = stability
    return row

DISPLAY_COLS = ['Mean Accuracy', 'Mean Recall', 'Mean Precision', 'Mean Specificity',
                'Mean F1', 'Mean AUC', 'Mean PR-AUC', 'Mean MCC', 'Std AUC', 'Stability Score']

def evaluate_stability(X, y, dataset_name):
    """Run Stage 1 stability evaluation with expanded metrics."""
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
    classifiers = get_base_classifiers()
    results = []
    fold_scores = {}

    for name, clf in classifiers.items():
        pipe = make_pipeline(clf, dataset_name)
        cv_results = cross_validate(pipe, X, y, cv=rskf, scoring=SCORING,
                                     return_train_score=False, n_jobs=-1)
        metrics = extract_metrics(cv_results)
        fold_scores[name] = metrics['roc_auc']
        results.append(metrics_to_row('Model', name, metrics))

    df = pd.DataFrame(results)
    df.to_csv(f'{DATASETS_DIR}/stability_results_{dataset_name}.csv', index=False)

    print(f'\n=== Stability Results — {dataset_name.upper()} ===')
    display_df = df[['Model'] + DISPLAY_COLS].copy()
    for col in DISPLAY_COLS:
        display_df[col] = df[col].map(lambda x: f'{x:.4f}')
    print(display_df.to_string(index=False))
    return df, fold_scores

print('Running stability evaluation for Heart Disease...')
heart_stability_df, heart_fold_scores = evaluate_stability(heart_X, heart_y, 'heart')

print('\nRunning stability evaluation for Pima Diabetes...')
pima_stability_df, pima_fold_scores = evaluate_stability(pima_X, pima_y, 'pima')

Running stability evaluation for Heart Disease...

=== Stability Results — HEART ===
  Model Mean Accuracy Mean Recall Mean Precision Mean Specificity Mean F1 Mean AUC Mean PR-AUC Mean MCC Std AUC Stability Score
     LR        0.8375      0.7914         0.8506           0.8769  0.8365   0.9059      0.9040   0.6766  0.0391         23.1836
    SVM        0.8342      0.7783         0.8521           0.8818  0.8331   0.8990      0.8973   0.6690  0.0378         23.7678
    kNN        0.8303      0.7885         0.8344           0.8659  0.8295   0.8907      0.8578   0.6596  0.0387         23.0077
     RF        0.8315      0.7769         0.8486           0.8780  0.8302   0.9047      0.8985   0.6644  0.0369         24.4944
XGBoost        0.8038      0.7639         0.8034           0.8377  0.8027   0.8787      0.8741   0.6076  0.0358         24.5689

Running stability evaluation for Pima Diabetes...

=== Stability Results — PIMA ===
  Model Mean Accuracy Mean Recall Mean Precision Mean Specific

In [ ]:
# --- Compute stability-score-based weights ---
def compute_weights(stability_df, selected_models=None):
    """Compute normalized weights from stability scores. If selected_models given, filter first."""
    df = stability_df.copy()
    if selected_models is not None:
        df = df[df['Model'].isin(selected_models)]
    raw = df.set_index('Model')['Stability Score']
    normalized = raw / raw.sum()
    return raw, normalized

def print_weights_table(stability_df, selected_models, label, dataset_name):
    """Print weight table for a set of models."""
    raw, norm = compute_weights(stability_df, selected_models)
    print(f'\n=== Weights — {dataset_name.upper()} | {label} ===')
    tbl = pd.DataFrame({'Raw Weight': raw, 'Normalized Weight': norm})
    tbl['Raw Weight'] = tbl['Raw Weight'].map(lambda x: f'{x:.4f}')
    tbl['Normalized Weight'] = tbl['Normalized Weight'].map(lambda x: f'{x:.4f}')
    print(tbl.to_string())

# All models weights (printed now; median-pruned weights printed after pruning stage)
for dname, sdf in [('heart', heart_stability_df), ('pima', pima_stability_df)]:
    print_weights_table(sdf, None, 'All Models', dname)


=== Weights — HEART | All Models ===
        Raw Weight Normalized Weight
Model                               
LR         23.1836            0.1948
SVM        23.7678            0.1997
kNN        23.0077            0.1933
RF         24.4944            0.2058
XGBoost    24.5689            0.2064

=== Weights — PIMA | All Models ===
        Raw Weight Normalized Weight
Model                               
LR         26.9698            0.2129
SVM        28.4534            0.2246
kNN        19.8053            0.1563
RF         25.4726            0.2010
XGBoost    26.0061            0.2052


## 5. Stage 2 — Pruning (Two Strategies)

Two selection strategies are compared:
- **Strategy 1 — Mean-based**: threshold = mean of all models' Mean/Std AUC
- **Strategy 2 — Median-based**: threshold = median of all models' Mean/Std AUC

Selection condition for both: `Mean AUC >= threshold  AND  Std AUC <= threshold`

In [8]:
def apply_pruning_strategy(stability_df, use_median=False):
    """Select models using mean or median thresholds. Returns selected list."""
    mean_aucs = stability_df['Mean AUC']
    std_aucs = stability_df['Std AUC']

    auc_thresh = mean_aucs.median() if use_median else mean_aucs.mean()
    std_thresh = std_aucs.median() if use_median else std_aucs.mean()

    relaxation = 0.0
    selected = []
    while len(selected) < 2:
        mask = (mean_aucs >= (auc_thresh - relaxation)) & (std_aucs <= (std_thresh + relaxation))
        selected = stability_df[mask]['Model'].tolist()
        if len(selected) < 2:
            relaxation += 0.005
            if relaxation > 0.1:
                selected = stability_df.nlargest(2, 'Stability Score')['Model'].tolist()
                break

    return selected, auc_thresh - relaxation, std_thresh + relaxation, relaxation


def prune_models_both_strategies(stability_df, dataset_name):
    """Apply both strategies, print results, and return selected model lists."""
    selected_mean, auc_t1, std_t1, relax1 = apply_pruning_strategy(stability_df, use_median=False)
    selected_median, auc_t2, std_t2, relax2 = apply_pruning_strategy(stability_df, use_median=True)

    for label, selected, auc_t, std_t, relax in [
        ('Strategy 1 — Mean-based Thresholds',   selected_mean,   auc_t1, std_t1, relax1),
        ('Strategy 2 — Median-based Thresholds', selected_median, auc_t2, std_t2, relax2),
    ]:
        print(f'\n=== Pruning — {dataset_name.upper()} | {label} ===')
        print(f'Mean AUC threshold: {auc_t:.4f}  |  Std AUC threshold: {std_t:.4f}', end='')
        print(f'  (relaxed by {relax:.3f})' if relax > 0 else '')
        for _, row in stability_df.iterrows():
            name = row['Model']
            tag = 'SELECTED' if name in selected else 'REJECTED'
            reasons = []
            if name not in selected:
                if row['Mean AUC'] < auc_t:
                    reasons.append(f'Mean AUC {row["Mean AUC"]:.4f} < {auc_t:.4f}')
                if row['Std AUC'] > std_t:
                    reasons.append(f'Std AUC {row["Std AUC"]:.4f} > {std_t:.4f}')
            suffix = f' — {"; ".join(reasons)}' if reasons else f' — Mean AUC={row["Mean AUC"]:.4f}, Std AUC={row["Std AUC"]:.4f}'
            print(f'  {tag}: {name}{suffix}')

    # Side-by-side comparison table
    print(f'\n=== Side-by-Side Comparison — {dataset_name.upper()} ===')
    comp = stability_df[['Model', 'Mean AUC', 'Std AUC']].copy()
    comp['Mean AUC'] = comp['Mean AUC'].map(lambda x: f'{x:.4f}')
    comp['Std AUC']  = comp['Std AUC'].map(lambda x: f'{x:.4f}')
    comp['Selected (Mean)']   = stability_df['Model'].apply(lambda m: 'YES' if m in selected_mean   else 'no')
    comp['Selected (Median)'] = stability_df['Model'].apply(lambda m: 'YES' if m in selected_median else 'no')
    print(comp.to_string(index=False))

    return selected_mean, selected_median


heart_selected_mean, heart_selected_median = prune_models_both_strategies(heart_stability_df, 'heart')
pima_selected_mean,  pima_selected_median  = prune_models_both_strategies(pima_stability_df,  'pima')


=== Pruning — HEART | Strategy 1 — Mean-based Thresholds ===
Mean AUC threshold: 0.8908  |  Std AUC threshold: 0.0427  (relaxed by 0.005)
  SELECTED: LR — Mean AUC=0.9059, Std AUC=0.0391
  SELECTED: SVM — Mean AUC=0.8990, Std AUC=0.0378
  REJECTED: kNN — Mean AUC 0.8907 < 0.8908
  SELECTED: RF — Mean AUC=0.9047, Std AUC=0.0369
  REJECTED: XGBoost — Mean AUC 0.8787 < 0.8908

=== Pruning — HEART | Strategy 2 — Median-based Thresholds ===
Mean AUC threshold: 0.8990  |  Std AUC threshold: 0.0378
  REJECTED: LR — Std AUC 0.0391 > 0.0378
  SELECTED: SVM — Mean AUC=0.8990, Std AUC=0.0378
  REJECTED: kNN — Mean AUC 0.8907 < 0.8990; Std AUC 0.0387 > 0.0378
  SELECTED: RF — Mean AUC=0.9047, Std AUC=0.0369
  REJECTED: XGBoost — Mean AUC 0.8787 < 0.8990

=== Side-by-Side Comparison — HEART ===
  Model Mean AUC Std AUC Selected (Mean) Selected (Median)
     LR   0.9059  0.0391             YES                no
    SVM   0.8990  0.0378             YES               YES
    kNN   0.8907  0.0387     

## 6. Stage 3 — Build & Evaluate All Methods

Evaluate the following methods with the same RepeatedStratifiedKFold (5×5):
1. Individual classifiers (reuse Stage 1)
2. Full Soft Voting Ensemble
3. Full Stacking Ensemble
4. Pruned Ensemble — Mean Threshold (default params)
5. Pruned Ensemble — Median Threshold (default params)

In [9]:
def evaluate_ensemble_metrics(X, y, ensemble_pipe, rskf):
    """Evaluate an ensemble pipeline and return metric arrays."""
    cv_results = cross_validate(ensemble_pipe, X, y, cv=rskf, scoring=SCORING,
                                 return_train_score=False, n_jobs=-1)
    return extract_metrics(cv_results)


def build_and_evaluate_ensembles(X, y, selected_mean, selected_median,
                                 stability_df, fold_scores, dataset_name):
    """Build and evaluate all ensemble methods, including both pruned strategies."""
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
    all_classifiers = get_base_classifiers()

    all_fold_scores = dict(fold_scores)
    results = []

    # 1. Individual classifiers — reuse from stability_df
    for _, row in stability_df.iterrows():
        r = dict(row)
        r['Method'] = f"{r.pop('Model')} (default)"
        results.append(r)

    # 2. Full Soft Voting Ensemble
    print('  Evaluating Full Soft Voting...')
    full_estimators = [(name, clf) for name, clf in all_classifiers.items()]
    metrics = evaluate_ensemble_metrics(X, y, make_ensemble_pipeline(
        VotingClassifier(estimators=full_estimators, voting='soft'), dataset_name), rskf)
    all_fold_scores['Full Soft Voting'] = metrics['roc_auc']
    results.append(metrics_to_row('Method', 'Full Soft Voting', metrics))

    # 3. Full Stacking Ensemble
    print('  Evaluating Full Stacking...')
    metrics = evaluate_ensemble_metrics(X, y, make_ensemble_pipeline(
        StackingClassifier(
            estimators=full_estimators,
            final_estimator=LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
            cv=5
        ), dataset_name), rskf)
    all_fold_scores['Full Stacking'] = metrics['roc_auc']
    results.append(metrics_to_row('Method', 'Full Stacking', metrics))

    # 4. Pruned Ensemble — Mean Threshold
    print(f'  Evaluating Pruned Ensemble (Mean Threshold) [{selected_mean}]...')
    pruned_est_mean = [(name, all_classifiers[name]) for name in selected_mean]
    metrics = evaluate_ensemble_metrics(X, y, make_ensemble_pipeline(
        VotingClassifier(estimators=pruned_est_mean, voting='soft'), dataset_name), rskf)
    all_fold_scores['Pruned Ensemble (Mean Threshold)'] = metrics['roc_auc']
    results.append(metrics_to_row('Method', 'Pruned Ensemble (Mean Threshold)', metrics))

    # 5. Pruned Ensemble — Median Threshold
    print(f'  Evaluating Pruned Ensemble (Median Threshold) [{selected_median}]...')
    pruned_est_median = [(name, all_classifiers[name]) for name in selected_median]
    metrics = evaluate_ensemble_metrics(X, y, make_ensemble_pipeline(
        VotingClassifier(estimators=pruned_est_median, voting='soft'), dataset_name), rskf)
    all_fold_scores['Pruned Ensemble (Median Threshold)'] = metrics['roc_auc']
    results.append(metrics_to_row('Method', 'Pruned Ensemble (Median Threshold)', metrics))

    return results, all_fold_scores


print('=== Stage 3 — Heart Disease ===')
heart_results, heart_all_fold_scores = build_and_evaluate_ensembles(
    heart_X, heart_y, heart_selected_mean, heart_selected_median,
    heart_stability_df, heart_fold_scores, 'heart')

print('\n=== Stage 3 — Pima Diabetes ===')
pima_results, pima_all_fold_scores = build_and_evaluate_ensembles(
    pima_X, pima_y, pima_selected_mean, pima_selected_median,
    pima_stability_df, pima_fold_scores, 'pima')

=== Stage 3 — Heart Disease ===
  Evaluating Full Soft Voting...
  Evaluating Full Stacking...
  Evaluating Pruned Ensemble (Mean Threshold) [['LR', 'SVM', 'RF']]...
  Evaluating Pruned Ensemble (Median Threshold) [['SVM', 'RF']]...

=== Stage 3 — Pima Diabetes ===
  Evaluating Full Soft Voting...
  Evaluating Full Stacking...
  Evaluating Pruned Ensemble (Mean Threshold) [['LR', 'SVM', 'RF']]...
  Evaluating Pruned Ensemble (Median Threshold) [['LR', 'SVM']]...


## 7. Stage 4 — Hyperparameter Tuning (Both Pruned Ensembles)

Tuning happens **after** pruning on neutral default params to avoid bias.
Both mean-threshold and median-threshold pruned ensembles are tuned independently.
If both strategies selected identical models, tuning is done once and reused.

In [10]:
PARAM_GRIDS = {
    'LR':      {'clf__C': [0.01, 0.1, 1, 10]},
    'SVM':     {'clf__C': [0.1, 1, 10], 'clf__gamma': ['scale', 'auto']},
    'kNN':     {'clf__n_neighbors': [3, 5, 7, 11]},
    'RF':      {'clf__n_estimators': [100, 200], 'clf__max_depth': [None, 5, 10]},
    'XGBoost': {'clf__n_estimators': [100, 200], 'clf__learning_rate': [0.05, 0.1]}
}


def tune_models(X, y, model_names, dataset_name, label):
    """Run GridSearchCV for each model in model_names. Returns dict of name -> tuned clf."""
    all_classifiers = get_base_classifiers()
    tuned = {}
    print(f'\n--- Tuning {label} models: {model_names} ---')
    for name in model_names:
        pipe = make_pipeline(all_classifiers[name], dataset_name)
        gs = GridSearchCV(pipe, PARAM_GRIDS[name], cv=3, scoring='roc_auc', n_jobs=-1)
        gs.fit(X, y)
        tuned[name] = gs.best_estimator_.named_steps['clf']
        print(f'  {name}: best params = {gs.best_params_}, best AUC = {gs.best_score_:.4f}')
    return tuned


def build_tuned_ensemble(X, y, tuned_clfs, model_names, dataset_name, method_label):
    """Build and evaluate a tuned pruned voting ensemble."""
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
    estimators = [(name, tuned_clfs[name]) for name in model_names]
    pipe = make_ensemble_pipeline(
        VotingClassifier(estimators=estimators, voting='soft'), dataset_name)
    print(f'  Evaluating {method_label}...')
    cv_results = cross_validate(pipe, X, y, cv=rskf, scoring=SCORING,
                                 return_train_score=False, n_jobs=-1)
    metrics = extract_metrics(cv_results)
    return metrics_to_row('Method', method_label, metrics), metrics['roc_auc']


def tune_both_pruned_ensembles(X, y, selected_mean, selected_median, dataset_name):
    """Tune both pruned ensembles. Reuses tuning if model sets are identical."""
    print(f'\n=== Hyperparameter Tuning — {dataset_name.upper()} ===')

    same_models = sorted(selected_mean) == sorted(selected_median)
    if same_models:
        print(f'NOTE: Mean and Median thresholds selected identical models: {selected_mean}')
        print('      Tuning once and reusing for both ensembles.')

    # Tune mean-threshold models
    tuned_mean = tune_models(X, y, selected_mean, dataset_name, 'Mean Threshold')
    mean_result, mean_auc = build_tuned_ensemble(
        X, y, tuned_mean, selected_mean, dataset_name,
        'Pruned Ensemble (Mean Threshold, Tuned)')

    if same_models:
        # Reuse same tuned classifiers
        median_result, median_auc = build_tuned_ensemble(
            X, y, tuned_mean, selected_median, dataset_name,
            'Pruned Ensemble (Median Threshold, Tuned)')
    else:
        # Tune median-threshold models independently
        # Only tune models not already tuned in the mean set
        models_to_tune = [m for m in selected_median if m not in tuned_mean]
        models_reused = [m for m in selected_median if m in tuned_mean]
        if models_reused:
            print(f'\n  Reusing tuned params from Mean Threshold for: {models_reused}')

        tuned_median = dict(tuned_mean)  # start with already-tuned models
        if models_to_tune:
            extra = tune_models(X, y, models_to_tune, dataset_name, 'Median Threshold (extra)')
            tuned_median.update(extra)

        median_result, median_auc = build_tuned_ensemble(
            X, y, tuned_median, selected_median, dataset_name,
            'Pruned Ensemble (Median Threshold, Tuned)')

    return mean_result, mean_auc, median_result, median_auc


heart_mean_tuned, heart_mean_tuned_auc, heart_median_tuned, heart_median_tuned_auc = \
    tune_both_pruned_ensembles(heart_X, heart_y, heart_selected_mean, heart_selected_median, 'heart')

pima_mean_tuned, pima_mean_tuned_auc, pima_median_tuned, pima_median_tuned_auc = \
    tune_both_pruned_ensembles(pima_X, pima_y, pima_selected_mean, pima_selected_median, 'pima')


=== Hyperparameter Tuning — HEART ===

--- Tuning Mean Threshold models: ['LR', 'SVM', 'RF'] ---
  LR: best params = {'clf__C': 0.01}, best AUC = 0.9046
  SVM: best params = {'clf__C': 0.1, 'clf__gamma': 'scale'}, best AUC = 0.9001
  RF: best params = {'clf__max_depth': 5, 'clf__n_estimators': 200}, best AUC = 0.9059
  Evaluating Pruned Ensemble (Mean Threshold, Tuned)...

  Reusing tuned params from Mean Threshold for: ['SVM', 'RF']
  Evaluating Pruned Ensemble (Median Threshold, Tuned)...

=== Hyperparameter Tuning — PIMA ===

--- Tuning Mean Threshold models: ['LR', 'SVM', 'RF'] ---
  LR: best params = {'clf__C': 0.1}, best AUC = 0.8367
  SVM: best params = {'clf__C': 0.1, 'clf__gamma': 'scale'}, best AUC = 0.8237
  RF: best params = {'clf__max_depth': 5, 'clf__n_estimators': 200}, best AUC = 0.8369
  Evaluating Pruned Ensemble (Mean Threshold, Tuned)...

  Reusing tuned params from Mean Threshold for: ['LR', 'SVM']
  Evaluating Pruned Ensemble (Median Threshold, Tuned)...


## 8. Stage 5 — Weighted Soft Voting (Stability-Score Weights)

Instead of equal soft voting, each model's contribution is weighted by its stability score.
Two variants: all 5 models weighted, and median-pruned models weighted.

In [11]:
# Print median-pruned weights (now that pruning has run)
for dname, sdf, sel in [('heart', heart_stability_df, heart_selected_median),
                         ('pima', pima_stability_df, pima_selected_median)]:
    print_weights_table(sdf, sel, 'Median Pruned', dname)


def weighted_vote(fitted_pipelines, weights, X):
    """Compute weighted average of predicted probabilities."""
    weighted_probs = np.zeros((len(X), 2))
    for pipe, w in zip(fitted_pipelines, weights):
        probs = pipe.predict_proba(X)
        weighted_probs += w * probs
    return weighted_probs


def evaluate_weighted_voting(X, y, model_names, stability_df, dataset_name, method_label):
    """Evaluate weighted soft voting via manual CV loop."""
    _, norm_weights = compute_weights(stability_df, model_names)
    weights_list = [norm_weights[name] for name in model_names]

    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
    all_classifiers = get_base_classifiers()

    # Collect per-fold metric scores
    fold_metrics = {m: [] for m in ['accuracy', 'recall', 'precision', 'specificity',
                                     'f1_weighted', 'roc_auc', 'pr_auc', 'mcc']}

    for train_idx, test_idx in rskf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Fit each model's pipeline independently
        fitted = []
        for name in model_names:
            pipe = make_pipeline(all_classifiers[name], dataset_name)
            pipe.fit(X_train, y_train)
            fitted.append(pipe)

        # Weighted vote
        probs = weighted_vote(fitted, weights_list, X_test)
        y_pred = np.argmax(probs, axis=1)
        y_prob = probs[:, 1]

        # Compute metrics
        fold_metrics['accuracy'].append(accuracy_score(y_test, y_pred))
        fold_metrics['recall'].append(recall_score(y_test, y_pred))
        fold_metrics['precision'].append(precision_score(y_test, y_pred, zero_division=0))
        fold_metrics['specificity'].append(specificity_score(y_test, y_pred))
        fold_metrics['f1_weighted'].append(f1_score(y_test, y_pred, average='weighted'))
        fold_metrics['roc_auc'].append(roc_auc_score(y_test, y_prob))
        fold_metrics['pr_auc'].append(average_precision_score(y_test, y_prob))
        fold_metrics['mcc'].append(matthews_corrcoef(y_test, y_pred))

    # Convert to numpy arrays
    metrics = {k: np.array(v) for k, v in fold_metrics.items()}
    result = metrics_to_row('Method', method_label, metrics)
    print(f'  {method_label}: Mean AUC = {metrics["roc_auc"].mean():.4f} +/- {metrics["roc_auc"].std():.4f}')
    return result, metrics['roc_auc']


# Evaluate weighted voting for both datasets
all_model_names = ['LR', 'SVM', 'kNN', 'RF', 'XGBoost']

print('\n=== Stage 5 — Weighted Voting ===')
print('\n--- Heart Disease ---')
heart_weighted_all, heart_weighted_all_auc = evaluate_weighted_voting(
    heart_X, heart_y, all_model_names, heart_stability_df, 'heart', 'Weighted Voting (All Models)')
heart_weighted_pruned, heart_weighted_pruned_auc = evaluate_weighted_voting(
    heart_X, heart_y, heart_selected_median, heart_stability_df, 'heart', 'Weighted Voting (Median Pruned)')

print('\n--- Pima Diabetes ---')
pima_weighted_all, pima_weighted_all_auc = evaluate_weighted_voting(
    pima_X, pima_y, all_model_names, pima_stability_df, 'pima', 'Weighted Voting (All Models)')
pima_weighted_pruned, pima_weighted_pruned_auc = evaluate_weighted_voting(
    pima_X, pima_y, pima_selected_median, pima_stability_df, 'pima', 'Weighted Voting (Median Pruned)')


=== Weights — HEART | Median Pruned ===
      Raw Weight Normalized Weight
Model                             
SVM      23.7678            0.4925
RF       24.4944            0.5075

=== Weights — PIMA | Median Pruned ===
      Raw Weight Normalized Weight
Model                             
LR       26.9698            0.4866
SVM      28.4534            0.5134

=== Stage 5 — Weighted Voting ===

--- Heart Disease ---
  Weighted Voting (All Models): Mean AUC = 0.9092 +/- 0.0366
  Weighted Voting (Median Pruned): Mean AUC = 0.9084 +/- 0.0368

--- Pima Diabetes ---
  Weighted Voting (All Models): Mean AUC = 0.8348 +/- 0.0312
  Weighted Voting (Median Pruned): Mean AUC = 0.8424 +/- 0.0287


## 9. Stage 6 — Alpha-Weighted Voting (Performance/Stability Tradeoff)

Uses min-max normalized Mean AUC and Std AUC with an explicit alpha parameter:
`W = α * Mean_norm + (1-α) * (1 - Std_norm)` where α=0.7 (70% performance, 30% stability).
Weights are recomputed inside each CV fold using training data only to avoid leakage.

In [12]:
ALPHA = 0.7

def compute_alpha_weights(mean_aucs, std_aucs, alpha=ALPHA):
    """Compute alpha-weighted normalized weights from Mean AUC and Std AUC arrays."""
    mean_norm = (mean_aucs - mean_aucs.min()) / (mean_aucs.max() - mean_aucs.min() + 1e-9)
    std_norm = (std_aucs - std_aucs.min()) / (std_aucs.max() - std_aucs.min() + 1e-9)
    raw = alpha * mean_norm + (1 - alpha) * (1 - std_norm)
    normalized = raw / (raw.sum() + 1e-9)
    return mean_norm, std_norm, raw, normalized


def compute_fold_alpha_weights(X_train, y_train, model_names, dataset_name, alpha=ALPHA):
    """Compute alpha weights from inner CV on training fold (no leakage)."""
    inner_cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=2, random_state=RANDOM_STATE)
    all_classifiers = get_base_classifiers()
    mean_aucs, std_aucs = [], []

    for name in model_names:
        pipe = make_pipeline(all_classifiers[name], dataset_name)
        scores = cross_validate(pipe, X_train, y_train, cv=inner_cv,
                                scoring='roc_auc', n_jobs=-1)['test_score']
        mean_aucs.append(scores.mean())
        std_aucs.append(scores.std())

    mean_aucs = np.array(mean_aucs)
    std_aucs = np.array(std_aucs)
    _, _, _, normalized = compute_alpha_weights(mean_aucs, std_aucs, alpha)
    return normalized


# Print global alpha weight breakdown (for reporting — actual CV uses per-fold weights)
def print_alpha_weights_table(stability_df, model_names, label, dataset_name):
    """Print alpha weight breakdown table for a set of models."""
    sdf = stability_df[stability_df['Model'].isin(model_names)].copy()
    mean_aucs = sdf['Mean AUC'].values
    std_aucs = sdf['Std AUC'].values
    names = sdf['Model'].values

    mean_norm, std_norm, raw, normalized = compute_alpha_weights(mean_aucs, std_aucs)

    print(f'\n=== Alpha Weights (α={ALPHA}) — {dataset_name.upper()} | {label} ===')
    tbl = pd.DataFrame({
        'Model': names, 'Mean AUC': mean_aucs, 'Std AUC': std_aucs,
        'Mean_norm': mean_norm, 'Std_norm': std_norm,
        'Raw Weight': raw, 'Normalized Weight': normalized
    }).set_index('Model')
    for col in tbl.columns:
        tbl[col] = tbl[col].map(lambda x: f'{x:.4f}')
    print(tbl.to_string())

all_model_names = ['LR', 'SVM', 'kNN', 'RF', 'XGBoost']
for dname, sdf in [('heart', heart_stability_df), ('pima', pima_stability_df)]:
    print_alpha_weights_table(sdf, all_model_names, 'All Models', dname)
for dname, sdf, sel in [('heart', heart_stability_df, heart_selected_median),
                         ('pima', pima_stability_df, pima_selected_median)]:
    print_alpha_weights_table(sdf, sel, 'Median Pruned', dname)


=== Alpha Weights (α=0.7) — HEART | All Models ===
        Mean AUC Std AUC Mean_norm Std_norm Raw Weight Normalized Weight
Model                                                                   
LR        0.9059  0.0391    1.0000   1.0000     0.7000            0.2465
SVM       0.8990  0.0378    0.7461   0.6221     0.6356            0.2238
kNN       0.8907  0.0387    0.4412   0.8904     0.3417            0.1203
RF        0.9047  0.0369    0.9550   0.3536     0.8624            0.3037
XGBoost   0.8787  0.0358    0.0000   0.0000     0.3000            0.1056

=== Alpha Weights (α=0.7) — PIMA | All Models ===
        Mean AUC Std AUC Mean_norm Std_norm Raw Weight Normalized Weight
Model                                                                   
LR        0.8366  0.0310    1.0000   0.1924     0.9423            0.3058
SVM       0.8258  0.0290    0.8091   0.0000     0.8664            0.2812
kNN       0.7802  0.0394    0.0000   1.0000     0.0000            0.0000
RF        0.8258  0.0

In [13]:
def evaluate_alpha_weighted_voting(X, y, model_names, dataset_name, method_label, alpha=ALPHA):
    """Evaluate alpha-weighted voting with per-fold weight recomputation."""
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
    all_classifiers = get_base_classifiers()

    fold_metrics = {m: [] for m in ['accuracy', 'recall', 'precision', 'specificity',
                                     'f1_weighted', 'roc_auc', 'pr_auc', 'mcc']}

    for fold_num, (train_idx, test_idx) in enumerate(rskf.split(X, y)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Recompute weights from inner CV on training fold (no leakage)
        weights = compute_fold_alpha_weights(X_train, y_train, model_names, dataset_name, alpha)

        # Fit each model on full training fold
        fitted = []
        for name in model_names:
            pipe = make_pipeline(all_classifiers[name], dataset_name)
            pipe.fit(X_train, y_train)
            fitted.append(pipe)

        # Weighted vote
        probs = weighted_vote(fitted, weights, X_test)
        y_pred = np.argmax(probs, axis=1)
        y_prob = probs[:, 1]

        fold_metrics['accuracy'].append(accuracy_score(y_test, y_pred))
        fold_metrics['recall'].append(recall_score(y_test, y_pred))
        fold_metrics['precision'].append(precision_score(y_test, y_pred, zero_division=0))
        fold_metrics['specificity'].append(specificity_score(y_test, y_pred))
        fold_metrics['f1_weighted'].append(f1_score(y_test, y_pred, average='weighted'))
        fold_metrics['roc_auc'].append(roc_auc_score(y_test, y_prob))
        fold_metrics['pr_auc'].append(average_precision_score(y_test, y_prob))
        fold_metrics['mcc'].append(matthews_corrcoef(y_test, y_pred))

    metrics = {k: np.array(v) for k, v in fold_metrics.items()}
    result = metrics_to_row('Method', method_label, metrics)
    print(f'  {method_label}: Mean AUC = {metrics["roc_auc"].mean():.4f} +/- {metrics["roc_auc"].std():.4f}')
    return result, metrics['roc_auc']


print('=== Stage 6 — Alpha-Weighted Voting ===')
print('\n--- Heart Disease ---')
heart_alpha_all, heart_alpha_all_auc = evaluate_alpha_weighted_voting(
    heart_X, heart_y, all_model_names, 'heart',
    f'Alpha-Weighted Voting (All Models, \u03b1={ALPHA})')
heart_alpha_pruned, heart_alpha_pruned_auc = evaluate_alpha_weighted_voting(
    heart_X, heart_y, heart_selected_median, 'heart',
    f'Alpha-Weighted Voting (Median Pruned, \u03b1={ALPHA})')

print('\n--- Pima Diabetes ---')
pima_alpha_all, pima_alpha_all_auc = evaluate_alpha_weighted_voting(
    pima_X, pima_y, all_model_names, 'pima',
    f'Alpha-Weighted Voting (All Models, \u03b1={ALPHA})')
pima_alpha_pruned, pima_alpha_pruned_auc = evaluate_alpha_weighted_voting(
    pima_X, pima_y, pima_selected_median, 'pima',
    f'Alpha-Weighted Voting (Median Pruned, \u03b1={ALPHA})')

=== Stage 6 — Alpha-Weighted Voting ===

--- Heart Disease ---
  Alpha-Weighted Voting (All Models, α=0.7): Mean AUC = 0.9077 +/- 0.0366
  Alpha-Weighted Voting (Median Pruned, α=0.7): Mean AUC = 0.9053 +/- 0.0366

--- Pima Diabetes ---
  Alpha-Weighted Voting (All Models, α=0.7): Mean AUC = 0.8383 +/- 0.0299
  Alpha-Weighted Voting (Median Pruned, α=0.7): Mean AUC = 0.8397 +/- 0.0297


## 10. Final Results Table

In [14]:
FINAL_DISPLAY_COLS = ['Mean Accuracy', 'Mean Recall', 'Mean Precision', 'Mean Specificity',
                      'Mean F1', 'Mean AUC', 'Mean PR-AUC', 'Mean MCC',
                      'Std Accuracy', 'Std Recall', 'Std Precision', 'Std Specificity',
                      'Std F1', 'Std AUC', 'Std PR-AUC', 'Std MCC']

def build_final_table(results_list, mean_tuned, median_tuned,
                      weighted_all, weighted_pruned,
                      alpha_all, alpha_pruned, dataset_name):
    """Combine all results into master DataFrame, highlight best values."""
    all_results = []
    for r in results_list:
        all_results.append(r)
        if r.get('Method') == 'Pruned Ensemble (Mean Threshold)':
            all_results.append(mean_tuned)
        elif r.get('Method') == 'Pruned Ensemble (Median Threshold)':
            all_results.append(median_tuned)

    # Append weighted and alpha-weighted voting results
    all_results.extend([weighted_all, weighted_pruned, alpha_all, alpha_pruned])

    df = pd.DataFrame(all_results)
    df = df.set_index('Method')

    cols_to_save = [c for c in FINAL_DISPLAY_COLS if c in df.columns and not df[c].isna().all()]
    df = df[cols_to_save]
    df.to_csv(f'{DATASETS_DIR}/final_results_{dataset_name}.csv')

    print(f'\n{"=" * 100}')
    print(f'FINAL RESULTS — {dataset_name.upper()}')
    print(f'{"=" * 100}')

    compact_cols = ['Mean Accuracy', 'Mean Recall', 'Mean Precision', 'Mean Specificity',
                    'Mean F1', 'Mean AUC', 'Std AUC', 'Mean PR-AUC', 'Mean MCC']
    compact_cols = [c for c in compact_cols if c in df.columns and not df[c].isna().all()]
    display_df = df[compact_cols].copy()

    for col in compact_cols:
        display_df[col] = df[col].map(lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A')

    for col in compact_cols:
        col_data = df[col].dropna()
        if col_data.empty:
            continue
        best_idx = col_data.idxmin() if 'Std' in col else col_data.idxmax()
        display_df.loc[best_idx, col] = display_df.loc[best_idx, col] + ' *'

    print(display_df.to_string())
    print('\n(* = best value in column)')
    return df

heart_final_df = build_final_table(heart_results, heart_mean_tuned, heart_median_tuned,
                                    heart_weighted_all, heart_weighted_pruned,
                                    heart_alpha_all, heart_alpha_pruned, 'heart')
pima_final_df = build_final_table(pima_results, pima_mean_tuned, pima_median_tuned,
                                   pima_weighted_all, pima_weighted_pruned,
                                   pima_alpha_all, pima_alpha_pruned, 'pima')


FINAL RESULTS — HEART
                                             Mean Accuracy Mean Recall Mean Precision Mean Specificity   Mean F1  Mean AUC   Std AUC Mean PR-AUC  Mean MCC
Method                                                                                                                                                    
LR (default)                                        0.8375    0.7914 *         0.8506           0.8769    0.8365    0.9059    0.0391      0.9040    0.6766
SVM (default)                                       0.8342      0.7783         0.8521           0.8818    0.8331    0.8990    0.0378      0.8973    0.6690
kNN (default)                                       0.8303      0.7885         0.8344           0.8659    0.8295    0.8907    0.0387      0.8578    0.6596
RF (default)                                        0.8315      0.7769         0.8486           0.8780    0.8302    0.9047    0.0369      0.8985    0.6644
XGBoost (default)                              

## 11. Visualizations

In [15]:
# --- Plot 1: Bar chart — ROC-AUC mean + error bars ---
def plot_auc_comparison(final_df, dataset_name):
    fig, ax = plt.subplots(figsize=(12, 6))
    methods = final_df.index.tolist()
    means = final_df['Mean AUC'].values
    stds = final_df['Std AUC'].values

    colors = sns.color_palette('viridis', len(methods))
    ax.bar(range(len(methods)), means, yerr=stds, capsize=4,
           color=colors, edgecolor='black', linewidth=0.5)

    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('ROC-AUC')
    ax.set_title(f'ROC-AUC Comparison — {dataset_name.upper()}')
    ax.set_ylim(0.5, 1.0)
    plt.tight_layout()
    path = f'{PLOTS_DIR}/plot_auc_comparison_{dataset_name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {path}')

plot_auc_comparison(heart_final_df, 'heart')
plot_auc_comparison(pima_final_df, 'pima')

Saved plots/plot_auc_comparison_heart.png
Saved plots/plot_auc_comparison_pima.png


In [16]:
# --- Plot 2: Heatmap — stability scores across both datasets ---
model_names = ['LR', 'SVM', 'kNN', 'RF', 'XGBoost']
heart_stab = heart_stability_df.set_index('Model')['Stability Score']
pima_stab = pima_stability_df.set_index('Model')['Stability Score']

heatmap_df = pd.DataFrame({
    'Heart Disease': [heart_stab[m] for m in model_names],
    'Pima Diabetes': [pima_stab[m] for m in model_names]
}, index=model_names)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(heatmap_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='gray')
ax.set_title('Stability Scores (Mean AUC / Std AUC) Across Datasets')
ax.set_ylabel('Model')
plt.tight_layout()
path = f'{PLOTS_DIR}/plot_stability_heatmap.png'
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved {path}')

Saved plots/plot_stability_heatmap.png


In [17]:
# --- Plot 3: Box plots — AUC distribution across folds ---
def plot_boxplot(fold_scores, mean_tuned_auc, median_tuned_auc,
                 weighted_all_auc, weighted_pruned_auc,
                 alpha_all_auc, alpha_pruned_auc, dataset_name):
    all_scores = dict(fold_scores)
    all_scores['Pruned (Mean, Tuned)'] = mean_tuned_auc
    all_scores['Pruned (Median, Tuned)'] = median_tuned_auc
    all_scores['Weighted (All)'] = weighted_all_auc
    all_scores['Weighted (Median Pruned)'] = weighted_pruned_auc
    all_scores['Alpha-Wt (All)'] = alpha_all_auc
    all_scores['Alpha-Wt (Median Pruned)'] = alpha_pruned_auc

    order = ['LR', 'SVM', 'kNN', 'RF', 'XGBoost',
             'Full Soft Voting', 'Full Stacking',
             'Pruned Ensemble (Mean Threshold)',
             'Pruned (Mean, Tuned)',
             'Pruned Ensemble (Median Threshold)',
             'Pruned (Median, Tuned)',
             'Weighted (All)',
             'Weighted (Median Pruned)',
             'Alpha-Wt (All)',
             'Alpha-Wt (Median Pruned)']

    data, labels = [], []
    for name in order:
        if name in all_scores:
            data.append(all_scores[name])
            labels.append(name)

    fig, ax = plt.subplots(figsize=(22, 6))
    bp = ax.boxplot(data, patch_artist=True, notch=True)

    colors = sns.color_palette('viridis', len(labels))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('ROC-AUC')
    ax.set_title(f'ROC-AUC Distribution Across Folds — {dataset_name.upper()}')
    plt.tight_layout()
    path = f'{PLOTS_DIR}/plot_boxplot_{dataset_name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {path}')

plot_boxplot(heart_all_fold_scores, heart_mean_tuned_auc, heart_median_tuned_auc,
             heart_weighted_all_auc, heart_weighted_pruned_auc,
             heart_alpha_all_auc, heart_alpha_pruned_auc, 'heart')
plot_boxplot(pima_all_fold_scores, pima_mean_tuned_auc, pima_median_tuned_auc,
             pima_weighted_all_auc, pima_weighted_pruned_auc,
             pima_alpha_all_auc, pima_alpha_pruned_auc, 'pima')

Saved plots/plot_boxplot_heart.png
Saved plots/plot_boxplot_pima.png


In [18]:
# --- Plot 4: Grouped bar chart — Recall vs Specificity ---
def plot_recall_specificity(final_df, dataset_name):
    methods = final_df.index.tolist()
    recall = final_df['Mean Recall'].values
    spec = final_df['Mean Specificity'].values

    x = np.arange(len(methods))
    width = 0.35

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(x - width/2, recall, width, label='Recall (Sensitivity)',
           color=sns.color_palette('viridis', 2)[0], edgecolor='black', linewidth=0.5)
    ax.bar(x + width/2, spec, width, label='Specificity',
           color=sns.color_palette('viridis', 2)[1], edgecolor='black', linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Score')
    ax.set_title(f'Recall vs Specificity — {dataset_name.upper()}')
    ax.set_ylim(0.0, 1.05)
    ax.legend()
    plt.tight_layout()
    path = f'{PLOTS_DIR}/plot_recall_spec_{dataset_name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {path}')

plot_recall_specificity(heart_final_df, 'heart')
plot_recall_specificity(pima_final_df, 'pima')

Saved plots/plot_recall_spec_heart.png
Saved plots/plot_recall_spec_pima.png


In [19]:
# --- Plot 5: MCC comparison bar chart ---
def plot_mcc(final_df, dataset_name):
    methods = final_df.index.tolist()
    mcc_vals = final_df['Mean MCC'].values

    fig, ax = plt.subplots(figsize=(12, 6))
    colors = sns.color_palette('viridis', len(methods))
    ax.bar(range(len(methods)), mcc_vals, color=colors, edgecolor='black', linewidth=0.5)

    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('MCC')
    ax.set_title(f'Matthews Correlation Coefficient — {dataset_name.upper()}')
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    path = f'{PLOTS_DIR}/plot_mcc_{dataset_name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {path}')

plot_mcc(heart_final_df, 'heart')
plot_mcc(pima_final_df, 'pima')

Saved plots/plot_mcc_heart.png
Saved plots/plot_mcc_pima.png


In [20]:
# --- Plot 6: Normalized weights per model ---
def plot_weights(stability_df, selected_median, dataset_name):
    all_model_names = stability_df['Model'].tolist()
    _, norm_all = compute_weights(stability_df, None)
    _, norm_pruned = compute_weights(stability_df, selected_median)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # All models
    ax = axes[0]
    colors = sns.color_palette('viridis', len(all_model_names))
    bars = ax.bar(all_model_names, [norm_all[m] for m in all_model_names],
                  color=colors, edgecolor='black', linewidth=0.5)
    ax.set_ylabel('Normalized Weight')
    ax.set_title('Weighted Voting (All Models)')
    for bar, m in zip(bars, all_model_names):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{norm_all[m]:.3f}', ha='center', va='bottom', fontsize=9)

    # Median pruned
    ax = axes[1]
    pruned_colors = sns.color_palette('viridis', len(selected_median))
    bars = ax.bar(selected_median, [norm_pruned[m] for m in selected_median],
                  color=pruned_colors, edgecolor='black', linewidth=0.5)
    ax.set_ylabel('Normalized Weight')
    ax.set_title('Weighted Voting (Median Pruned)')
    for bar, m in zip(bars, selected_median):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{norm_pruned[m]:.3f}', ha='center', va='bottom', fontsize=9)

    fig.suptitle(f'Stability-Score Weights — {dataset_name.upper()}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    path = f'{PLOTS_DIR}/plot_weights_{dataset_name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {path}')

plot_weights(heart_stability_df, heart_selected_median, 'heart')
plot_weights(pima_stability_df, pima_selected_median, 'pima')

Saved plots/plot_weights_heart.png
Saved plots/plot_weights_pima.png


In [21]:
# --- Plot 7: Weight comparison across strategies ---
def plot_weight_comparison(stability_df, selected_median, dataset_name):
    all_names = stability_df['Model'].tolist()

    # Stability-score weights (all models)
    _, stab_norm_all = compute_weights(stability_df, None)

    # Alpha weights (all models) — global (for display)
    mean_aucs = stability_df['Mean AUC'].values
    std_aucs = stability_df['Std AUC'].values
    _, _, _, alpha_norm_all = compute_alpha_weights(mean_aucs, std_aucs)

    # Alpha weights (median pruned) — normalized within pruned set
    sdf_pruned = stability_df[stability_df['Model'].isin(selected_median)]
    _, _, _, alpha_norm_pruned = compute_alpha_weights(sdf_pruned['Mean AUC'].values,
                                                       sdf_pruned['Std AUC'].values)

    x = np.arange(len(all_names))
    width = 0.25

    fig, ax = plt.subplots(figsize=(14, 6))
    colors = sns.color_palette('viridis', 3)

    # Bar 1: Stability weights
    ax.bar(x - width, [stab_norm_all[m] for m in all_names], width,
           label='Stability Score (W=Mean/Std)', color=colors[0], edgecolor='black', linewidth=0.5)

    # Bar 2: Alpha weights all
    ax.bar(x, list(alpha_norm_all), width,
           label=f'Alpha All (\u03b1={ALPHA})', color=colors[1], edgecolor='black', linewidth=0.5)

    # Bar 3: Alpha weights pruned (only for selected models)
    alpha_pruned_vals = []
    pruned_names = sdf_pruned['Model'].tolist()
    pruned_idx = 0
    for m in all_names:
        if m in pruned_names:
            alpha_pruned_vals.append(alpha_norm_pruned[pruned_idx])
            pruned_idx += 1
        else:
            alpha_pruned_vals.append(0)
    ax.bar(x + width, alpha_pruned_vals, width,
           label=f'Alpha Pruned (\u03b1={ALPHA})', color=colors[2], edgecolor='black', linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(all_names, fontsize=10)
    ax.set_ylabel('Normalized Weight')
    ax.set_title(f'Weight Comparison Across Strategies — {dataset_name.upper()}')
    ax.legend()
    plt.tight_layout()
    path = f'{PLOTS_DIR}/plot_weight_comparison_{dataset_name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {path}')

plot_weight_comparison(heart_stability_df, heart_selected_median, 'heart')
plot_weight_comparison(pima_stability_df, pima_selected_median, 'pima')

Saved plots/plot_weight_comparison_heart.png
Saved plots/plot_weight_comparison_pima.png


## 12. Comparative Analysis

In [22]:
def print_analysis(final_df, selected_mean, selected_median, stability_df, dataset_name):
    """Print structured analysis answering key comparison questions."""
    print(f'\n{"=" * 80}')
    print(f'ANALYSIS — {dataset_name.upper()}')
    print(f'{"=" * 80}')

    methods = final_df.index

    # Q1: Did strategies select different models?
    same = sorted(selected_mean) == sorted(selected_median)
    print(f'\n1. Model Selection Differences:')
    print(f'   Mean Threshold selected:   {selected_mean}')
    print(f'   Median Threshold selected: {selected_median}')
    if same:
        print(f'   -> SAME models selected by both strategies.')
    else:
        only_mean = set(selected_mean) - set(selected_median)
        only_median = set(selected_median) - set(selected_mean)
        if only_mean:
            print(f'   -> Only in Mean strategy:   {sorted(only_mean)}')
        if only_median:
            print(f'   -> Only in Median strategy: {sorted(only_median)}')

    # Q2: Did tuning improve performance?
    print(f'\n2. Tuning Impact (ROC-AUC):')
    for strategy in ['Mean Threshold', 'Median Threshold']:
        default_name = f'Pruned Ensemble ({strategy})'
        tuned_name = f'Pruned Ensemble ({strategy}, Tuned)'
        if default_name in methods and tuned_name in methods:
            default_auc = final_df.loc[default_name, 'Mean AUC']
            tuned_auc = final_df.loc[tuned_name, 'Mean AUC']
            diff = tuned_auc - default_auc
            direction = 'improved' if diff > 0 else ('unchanged' if diff == 0 else 'decreased')
            print(f'   {strategy}: {default_auc:.4f} -> {tuned_auc:.4f} ({diff:+.4f}, {direction})')

    # Q3: Which pruning strategy performed better?
    print(f'\n3. Strategy Comparison (ROC-AUC):')
    for stage, suffix in [('Before tuning', ''), ('After tuning', ', Tuned')]:
        mean_name = f'Pruned Ensemble (Mean Threshold{suffix})'
        median_name = f'Pruned Ensemble (Median Threshold{suffix})'
        if mean_name in methods and median_name in methods:
            mean_auc = final_df.loc[mean_name, 'Mean AUC']
            median_auc = final_df.loc[median_name, 'Mean AUC']
            diff = mean_auc - median_auc
            winner = 'Mean' if diff > 0 else ('Median' if diff < 0 else 'Tie')
            print(f'   {stage}: Mean={mean_auc:.4f}, Median={median_auc:.4f} -> {winner} wins ({abs(diff):.4f})')

    # Q4: Highest ROC-AUC overall?
    print(f'\n4. Best ROC-AUC:')
    best_auc_method = final_df['Mean AUC'].idxmax()
    best_auc_val = final_df['Mean AUC'].max()
    print(f'   {best_auc_method} = {best_auc_val:.4f}')

    # Q5: Best MCC overall?
    print(f'\n5. Best MCC:')
    best_mcc_method = final_df['Mean MCC'].idxmax()
    best_mcc_val = final_df['Mean MCC'].max()
    print(f'   {best_mcc_method} = {best_mcc_val:.4f}')

    # Q6: Did pruned ensembles outperform full ensembles?
    print(f'\n6. Pruned vs Full Ensembles (ROC-AUC):')
    full_voting_auc = final_df.loc['Full Soft Voting', 'Mean AUC']
    full_stacking_auc = final_df.loc['Full Stacking', 'Mean AUC']
    best_full = max(full_voting_auc, full_stacking_auc)
    full_label = 'Full Soft Voting' if full_voting_auc >= full_stacking_auc else 'Full Stacking'

    pruned_methods = [m for m in methods if 'Pruned' in m]
    best_pruned_method = final_df.loc[pruned_methods, 'Mean AUC'].idxmax()
    best_pruned_auc = final_df.loc[best_pruned_method, 'Mean AUC']

    diff = best_pruned_auc - best_full
    print(f'   Best full ensemble:   {full_label} = {best_full:.4f}')
    print(f'   Best pruned ensemble: {best_pruned_method} = {best_pruned_auc:.4f}')
    if diff > 0:
        print(f'   -> Pruned ensemble OUTPERFORMS full ensemble by {diff:.4f}')
    elif diff == 0:
        print(f'   -> Tied.')
    else:
        print(f'   -> Full ensemble outperforms by {abs(diff):.4f}')

    # Q7: Did weighted voting outperform equal soft voting?
    print(f'\n7. Weighted vs Equal Soft Voting (ROC-AUC):')
    weighted_all_auc = final_df.loc['Weighted Voting (All Models)', 'Mean AUC']
    equal_full_auc = final_df.loc['Full Soft Voting', 'Mean AUC']
    diff = weighted_all_auc - equal_full_auc
    winner = 'Weighted' if diff > 0 else ('Equal' if diff < 0 else 'Tie')
    print(f'   Full Equal Voting:    {equal_full_auc:.4f}')
    print(f'   Full Weighted Voting: {weighted_all_auc:.4f} ({diff:+.4f}, {winner} wins)')

    weighted_pruned_auc = final_df.loc['Weighted Voting (Median Pruned)', 'Mean AUC']
    equal_pruned_auc = final_df.loc['Pruned Ensemble (Median Threshold)', 'Mean AUC']
    diff2 = weighted_pruned_auc - equal_pruned_auc
    winner2 = 'Weighted' if diff2 > 0 else ('Equal' if diff2 < 0 else 'Tie')
    print(f'   Pruned Equal Voting:    {equal_pruned_auc:.4f}')
    print(f'   Pruned Weighted Voting: {weighted_pruned_auc:.4f} ({diff2:+.4f}, {winner2} wins)')

    # Q8: Did weighted pruned outperform equal pruned?
    print(f'\n8. Weighted Pruned vs Equal Pruned (ROC-AUC):')
    print(f'   Equal Pruned (Median):    {equal_pruned_auc:.4f}')
    print(f'   Weighted Pruned (Median): {weighted_pruned_auc:.4f} ({diff2:+.4f})')

    # Q9: Highest-weighted model
    print(f'\n9. Highest-Weighted Model (Stability Score):')
    _, norm_all = compute_weights(stability_df, None)
    top_model = norm_all.idxmax()
    top_weight = norm_all.max()
    stab_rank_model = stability_df.set_index('Model')['Stability Score'].idxmax()
    print(f'   {top_model} (weight={top_weight:.4f})')
    match = 'YES' if top_model == stab_rank_model else 'NO'
    print(f'   Matches highest stability score? {match} (top stability: {stab_rank_model})')

    # Q10: Weighted all vs weighted pruned
    print(f'\n10. Weighted All vs Weighted Pruned (ROC-AUC):')
    diff3 = weighted_all_auc - weighted_pruned_auc
    winner3 = 'All' if diff3 > 0 else ('Pruned' if diff3 < 0 else 'Tie')
    print(f'    Weighted All:    {weighted_all_auc:.4f}')
    print(f'    Weighted Pruned: {weighted_pruned_auc:.4f} ({diff3:+.4f}, {winner3} wins)')

    # Q11: Alpha vs Stability weights — numerical differences
    alpha_all_label = f'Alpha-Weighted Voting (All Models, \u03b1={ALPHA})'
    alpha_pruned_label = f'Alpha-Weighted Voting (Median Pruned, \u03b1={ALPHA})'
    print(f'\n11. Alpha vs Stability Weights — Numerical Differences:')
    # Compare all-model weights
    _, stab_w = compute_weights(stability_df, None)
    mean_aucs = stability_df['Mean AUC'].values
    std_aucs = stability_df['Std AUC'].values
    _, _, _, alpha_w = compute_alpha_weights(mean_aucs, std_aucs)
    model_names = stability_df['Model'].tolist()
    print(f'    {"Model":<10} {"Stability Wt":>14} {"Alpha Wt":>14} {"Diff":>10} {"Direction":>12}')
    for i, m in enumerate(model_names):
        sw = stab_w[m]
        aw = alpha_w[i]
        d = aw - sw
        direction = 'gained' if d > 0 else ('lost' if d < 0 else 'same')
        print(f'    {m:<10} {sw:>14.4f} {aw:>14.4f} {d:>+10.4f} {direction:>12}')

    # Q12: Did alpha-weighted outperform stability-weighted?
    print(f'\n12. Alpha-Weighted vs Stability-Weighted (ROC-AUC):')
    alpha_all_auc = final_df.loc[alpha_all_label, 'Mean AUC']
    stab_all_auc = final_df.loc['Weighted Voting (All Models)', 'Mean AUC']
    diff = alpha_all_auc - stab_all_auc
    winner = 'Alpha' if diff > 0 else ('Stability' if diff < 0 else 'Tie')
    print(f'    All Models:  Stability={stab_all_auc:.4f}, Alpha={alpha_all_auc:.4f} ({diff:+.4f}, {winner})')

    alpha_pr_auc = final_df.loc[alpha_pruned_label, 'Mean AUC']
    stab_pr_auc = final_df.loc['Weighted Voting (Median Pruned)', 'Mean AUC']
    diff2 = alpha_pr_auc - stab_pr_auc
    winner2 = 'Alpha' if diff2 > 0 else ('Stability' if diff2 < 0 else 'Tie')
    print(f'    Median Pruned: Stability={stab_pr_auc:.4f}, Alpha={alpha_pr_auc:.4f} ({diff2:+.4f}, {winner2})')

    # Q13: Alpha pruned vs alpha all
    print(f'\n13. Alpha-Weighted Pruned vs Alpha-Weighted All (ROC-AUC):')
    diff3 = alpha_pr_auc - alpha_all_auc
    winner3 = 'Pruned' if diff3 > 0 else ('All' if diff3 < 0 else 'Tie')
    print(f'    Alpha All:    {alpha_all_auc:.4f}')
    print(f'    Alpha Pruned: {alpha_pr_auc:.4f} ({diff3:+.4f}, {winner3} wins)')

    # Q14: Best overall across all ensemble strategies
    print(f'\n14. Best Overall Across All Ensemble Strategies:')
    ensemble_methods = [m for m in methods if m not in
                        [f'{n} (default)' for n in ['LR','SVM','kNN','RF','XGBoost']]]
    best_ens_auc_method = final_df.loc[ensemble_methods, 'Mean AUC'].idxmax()
    best_ens_auc_val = final_df.loc[best_ens_auc_method, 'Mean AUC']
    best_ens_mcc_method = final_df.loc[ensemble_methods, 'Mean MCC'].idxmax()
    best_ens_mcc_val = final_df.loc[best_ens_mcc_method, 'Mean MCC']

    def categorize(method_name):
        tags = []
        if 'Pruned' in method_name or 'Median Pruned' in method_name:
            tags.append('pruned')
        else:
            tags.append('full')
        if 'Alpha' in method_name:
            tags.append('alpha-weighted')
        elif 'Weighted' in method_name:
            tags.append('stability-weighted')
        elif 'Tuned' in method_name:
            tags.append('tuned equal-voting')
        else:
            tags.append('equal-voting')
        return ', '.join(tags)

    print(f'    Best ROC-AUC: {best_ens_auc_method} = {best_ens_auc_val:.4f} ({categorize(best_ens_auc_method)})')
    print(f'    Best MCC:     {best_ens_mcc_method} = {best_ens_mcc_val:.4f} ({categorize(best_ens_mcc_method)})')


print_analysis(heart_final_df, heart_selected_mean, heart_selected_median, heart_stability_df, 'heart')
print_analysis(pima_final_df, pima_selected_mean, pima_selected_median, pima_stability_df, 'pima')


ANALYSIS — HEART

1. Model Selection Differences:
   Mean Threshold selected:   ['LR', 'SVM', 'RF']
   Median Threshold selected: ['SVM', 'RF']
   -> Only in Mean strategy:   ['LR']

2. Tuning Impact (ROC-AUC):
   Mean Threshold: 0.9101 -> 0.9100 (-0.0001, decreased)
   Median Threshold: 0.9083 -> 0.9092 (+0.0009, improved)

3. Strategy Comparison (ROC-AUC):
   Before tuning: Mean=0.9101, Median=0.9083 -> Mean wins (0.0018)
   After tuning: Mean=0.9100, Median=0.9092 -> Mean wins (0.0008)

4. Best ROC-AUC:
   Pruned Ensemble (Mean Threshold) = 0.9101

5. Best MCC:
   Pruned Ensemble (Mean Threshold, Tuned) = 0.6783

6. Pruned vs Full Ensembles (ROC-AUC):
   Best full ensemble:   Full Stacking = 0.9097
   Best pruned ensemble: Pruned Ensemble (Mean Threshold) = 0.9101
   -> Pruned ensemble OUTPERFORMS full ensemble by 0.0004

7. Weighted vs Equal Soft Voting (ROC-AUC):
   Full Equal Voting:    0.9096
   Full Weighted Voting: 0.9092 (-0.0004, Equal wins)
   Pruned Equal Voting:    0.908

## 13. Output Files Summary

In [23]:
import os

expected_files = [
    f'{DATASETS_DIR}/heart_disease_clean.csv',
    f'{DATASETS_DIR}/pima_diabetes_clean.csv',
    f'{DATASETS_DIR}/stability_results_heart.csv',
    f'{DATASETS_DIR}/stability_results_pima.csv',
    f'{DATASETS_DIR}/final_results_heart.csv',
    f'{DATASETS_DIR}/final_results_pima.csv',
    f'{PLOTS_DIR}/plot_auc_comparison_heart.png',
    f'{PLOTS_DIR}/plot_auc_comparison_pima.png',
    f'{PLOTS_DIR}/plot_stability_heatmap.png',
    f'{PLOTS_DIR}/plot_boxplot_heart.png',
    f'{PLOTS_DIR}/plot_boxplot_pima.png',
    f'{PLOTS_DIR}/plot_recall_spec_heart.png',
    f'{PLOTS_DIR}/plot_recall_spec_pima.png',
    f'{PLOTS_DIR}/plot_mcc_heart.png',
    f'{PLOTS_DIR}/plot_mcc_pima.png',
    f'{PLOTS_DIR}/plot_weights_heart.png',
    f'{PLOTS_DIR}/plot_weights_pima.png',
    f'{PLOTS_DIR}/plot_weight_comparison_heart.png',
    f'{PLOTS_DIR}/plot_weight_comparison_pima.png',
]

print('=== Generated Files ===')
for f in expected_files:
    exists = os.path.isfile(f)
    size = os.path.getsize(f) if exists else 0
    status = f'{size:,} bytes' if exists else 'MISSING'
    print(f'  {f:60s} {status}')

print('\nDone! All stages completed successfully.')

=== Generated Files ===
  datasets/heart_disease_clean.csv                             12,475 bytes
  datasets/pima_diabetes_clean.csv                             23,290 bytes
  datasets/stability_results_heart.csv                         1,889 bytes
  datasets/stability_results_pima.csv                          1,875 bytes
  datasets/final_results_heart.csv                             5,280 bytes
  datasets/final_results_pima.csv                              5,220 bytes
  plots/plot_auc_comparison_heart.png                          121,660 bytes
  plots/plot_auc_comparison_pima.png                           121,913 bytes
  plots/plot_stability_heatmap.png                             47,062 bytes
  plots/plot_boxplot_heart.png                                 150,719 bytes
  plots/plot_boxplot_pima.png                                  146,869 bytes
  plots/plot_recall_spec_heart.png                             130,942 bytes
  plots/plot_recall_spec_pima.png                              